# T16-T20 read-only server audit
Remote Python source below ran on p25 using SSH stdin. This cell is for inspection; running it locally requires the remote filesystem. No generation jobs were started or changed.

In [1]:
from pathlib import Path
from collections import Counter
import json,datetime,os
out={'captured_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
root=Path('/data_cached/wangzijun_p25/t16_formal_52800_20260909/offline')
out['genesis']={}
for name in ['t16_20260910_v1','t17_t18_engineering_20260910_v1','t17_t18_boundaries_20260910_v1']:
 d=root/name
 docs={}
 for n in ['summary.json','release_audit.json','formal_manifest.json','T17_engineering_result.json','T18_engineering_result.json']:
  p=d/n
  if p.exists():docs[n]=json.loads(p.read_text())
 vs=list((d/'videos').rglob('*.mp4')) if (d/'videos').exists() else []
 grouped={}
 for p in vs:grouped.setdefault(str(p.parent),set()).add(p.stem)
 out['genesis'][name]={'documents':docs,'mp4_files':len(vs),'complete_camera_directories':sum({'cam01','cam02','cam03'}<=v for v in grouped.values()),'zero_byte_mp4':sum(p.stat().st_size==0 for p in vs),'latest_video_mtime_utc':datetime.datetime.fromtimestamp(max((p.stat().st_mtime for p in vs),default=0),datetime.timezone.utc).isoformat()}
isaac=Path('/data_cached/wangzijun_p25/isaac_out/Isaac_T16-T19_Dataset')
out['isaac']={}
for event in ['T16','T17','T18','T19']:
 d=isaac/event; folders=[p for p in d.iterdir() if p.is_dir()]
 qa=Counter();classification=Counter();formal=0;missing=0;mp4=0;complete=0
 for folder in folders:
  vs=[folder/(cam+'.mp4') for cam in ['cam01','cam02','cam03']]
  mp4+=sum(p.exists() and p.stat().st_size>0 for p in vs);complete+=all(p.exists() and p.stat().st_size>0 for p in vs)
  cp=folder/'metadata/complete.json'
  if cp.exists():
   c=json.loads(cp.read_text());qa[c.get('status','missing_status')]+=1;formal+=c.get('formal_videos',0)
  else:missing+=1
  cp=folder/'config.json'
  if cp.exists():classification[json.loads(cp.read_text()).get('classification','missing')]+=1
 out['isaac'][event]={'sample_dirs':len(folders),'nonempty_camera_mp4':mp4,'complete_three_views':complete,'qa':dict(qa),'classification':dict(classification),'formal_videos_in_completion_records':formal,'missing_completion':missing}
print(json.dumps(out))


{"captured_utc": "2026-10-10T11:17:49.026873+00:00", "genesis": {"t16_20260910_v1": {"documents": {"summary.json": {"run": "t16_20260910_v1", "phase": "NEEDS_REPAIR", "target_trajectories": 880, "target_videos": 2640, "gpus": ["0", "1", "2", "3"], "active_jobs": {}, "states": {"g01_moon": {"phase": "COMPLETE"}, "g02_mars": {"phase": "COMPLETE"}, "g03_earth": {"phase": "COMPLETE"}, "g04_high": {"phase": "SEARCH_FULL_CHAIN", "tested": 2, "latest_depth": 1, "latest_qa": "PASS"}, "g05_very_high": {"phase": "COMPLETE"}, "preproduction": {"phase": "COMPLETE"}, "bg03_neutral_studio__g01_moon": {"phase": "COMPLETE"}, "bg01_home_ase__g01_moon": {"phase": "COMPLETE"}, "bg02_physics_lab__g01_moon": {"phase": "COMPLETE"}, "bg04_library_shelf__g01_moon": {"phase": "COMPLETE"}, "bg03_neutral_studio__g02_mars": {"phase": "COMPLETE"}, "bg01_home_ase__g02_mars": {"phase": "COMPLETE"}, "bg02_physics_lab__g02_mars": {"phase": "COMPLETE"}, "bg04_library_shelf__g02_mars": {"phase": "COMPLETE"}, "bg03_neutr